# 4.5. Interim Stage (4.5): Region Data Enrichment

**Documentation:** [Notebook guide](README.md) · [Stage data description](../../data/data-pipeline/stage_04_5/README_data.md)


> ## ⚠️ This stage is already completed — you can skip it
>
> The pre-built region lookup table **`data/data-pipeline/stage_04_5/region_db.pkl`**
> covers all ~22,000 unique region strings found in the full 2021–2025 dataset.
> Stage 5 reads this file directly — no reprocessing is required.
>
> **Only rerun this notebook if** you have added new vacancy data that contains
> region strings not yet present in `region_db.pkl` (i.e. strings that return
> no match during Stage 5 region joining).
>
> **Requires an OpenAI API key** if you do rerun. Set `OPENAI_API_KEY` in `notebooks/data-pipeline/.env`.

## 4.5.1. Description

Although not a core classification or extraction stage, region enrichment plays a crucial role in improving the quality and usability of the geographic metadata associated with each job advertisement. Many original region fields are unstructured, multilingual, or ambiguous. This step standardizes, expands, and geolocates this information.

## 4.5.2. Load packages and configuration

Enable autoreload, add `code/` to the Python path, and import standard libraries (`general`, `pandas`, `json`, `os`).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
from pipeline_bootstrap import configure_pipeline
configure_pipeline()
import general as g
g.clean_memory()
import pandas as pd
import json

Import `stage1` (for region/click data helpers) and `stage3` (reused for `read_classification_prompt()` and `read_classification_schema()` which work identically for the region enrichment prompt and schema).

In [ ]:
import stage1 as st1
import stage3 as st3

If you added new data to pipeline - rename stage4_5/region_db.pkl to stage4_5/region_db_PREV.pkl to work with new added region only

Load the previous region DB snapshot (`region_db_PREV.pkl`) if it exists. This baseline is used to identify only the **new** region strings that need to be submitted to the API — strings already in the previous DB are skipped. Display the loaded DataFrame to confirm shape and columns (`original`, `region`).

In [ ]:
g.Config.STAGE4_5_REGION_PREV_DB_PATH

In [ ]:
# Load previous region DB snapshot if it exists; None means all regions are treated as new
if os.path.exists(g.Config.STAGE4_5_REGION_PREV_DB_PATH):
    prev_db = pd.read_pickle(g.Config.STAGE4_5_REGION_PREV_DB_PATH)
else:
    prev_db = None
prev_db

Load the current merged region DB (`region_db.pkl`) and deduplicate on `original` to ensure each raw region string appears only once. The shape indicates how many unique region strings are already standardised.

In [ ]:
region_df = pd.read_pickle(g.Config.STAGE4_5_REGION_DB_PATH)
#region_df.to_pickle(Config.STAGE4_5_REGION_DB_PATH)

In [ ]:
region_df.drop_duplicates(subset=['original'], keep='first', inplace=True)
region_df.shape

## 4.5.3. Mergin all region data files into one

Helper function `merge_pkl_files()` reads all `.pkl` files in the given folder and concatenates them into a single DataFrame. Used to merge all per-day id/region/click count files produced by Stage 1 into one flat table.

In [ ]:
def merge_pkl_files(folder_path):
    pkl_files = [f for f in os.listdir(folder_path) if f.endswith('.pkl')]
    region_df = pd.read_pickle(os.path.join(folder_path,pkl_files[0]))
    for f in pkl_files[1:]:
        id_df = pd.read_pickle(os.path.join( folder_path, f))
        region_df = pd.concat([region_df, id_df], ignore_index=True)
        print(f"Merged {f} with {region_df.shape}")
    return region_df

Merge all Stage 1 id/region/click pickles from `data/data-pipeline/stage_01/id_region_click/` into a single DataFrame `id_dfx`. Each row contains a vacancy `id`, its raw `region` string, and `number_of_clicks`. Display the result to confirm the full scope of unique region strings in the dataset.

In [ ]:
id_dfx = merge_pkl_files(g.Config.STAGE1_ID_REGION_NCLICK_PATH)
id_dfx

Sanity check — count vacancies with a null `id`. Should be 0 for a clean dataset.

In [ ]:
id_dfx["id"].isnull().sum()

Build `prev_list` — the set of region strings already standardised in the previous DB snapshot. Used as a filter in the next cell to find only the truly new strings.

In [ ]:
prev_list = []
if prev_db is not None:
    prev_list = prev_db["original"].tolist()

Compute `new_regions` — the set difference between all region strings in the dataset and those already standardised. These are the strings that need to be submitted to the OpenAI Batch API. Print to confirm the list is reasonable before proceeding.

In [ ]:
new_regions = list(set(id_dfx["region"]) - set(prev_list))
print(new_regions)

Wrap the new region strings into a DataFrame and save it as `region_db_NEW.pkl` — this is the input for the Batch API calls below. Then reload and inspect it to confirm contents before building batch files.

In [ ]:
new_regions_db = pd.DataFrame(new_regions, columns=['original']).reset_index(drop=True)
new_regions_db.to_pickle(g.Config.STAGE4_5_REGION_NEW_DB_PATH)

In [ ]:
new_regions_db = pd.read_pickle(g.Config.STAGE4_5_REGION_NEW_DB_PATH)
new_regions_db.shape

In [ ]:
new_regions_db

## 4.5.4. Create batch files for API

Initialise the OpenAI client using the API key from `Config`. Required for uploading batch files and creating batch jobs.

In [ ]:
# Verify API key is loaded (do not print the actual value)
print("OPENAI_API_KEY loaded:", bool(g.Config.OPENAI_API_KEY))

In [ ]:
from openai import OpenAI
client = OpenAI(api_key=g.Config.OPENAI_API_KEY)

Function `write_region_batch_file()` builds the JSONL batch input file for the OpenAI Batch API. Each line contains one API request with:
- the raw `original` region string as the user message
- the region selection function-calling schema (`select_region_schema.json`)
- the standardisation prompt (`select_region_prompt.txt`)

The file is named `{from}_{to}_{date}.jsonl` to track which slice of new regions it covers. Because the Batch API has a 50,000-request limit per job, large datasets are split into numbered chunks.

In [ ]:
def write_region_batch_file(in_batch_path, subname, region_data_db, response_schema, request_prompt, model = "gpt-4o"):

    batch_path = os.path.join(in_batch_path, f"{subname}.jsonl")
    with open(batch_path, "w", encoding="utf-8") as fout:
        for _, r in region_data_db.iterrows():
            payload = {
                "original": r.loc["original"]
            }
            fout.write(json.dumps({
                "custom_id": "task-id-" + str(_),
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "temperature": 0,
                    "model": model,
                    "response_format": { "type": "json_object" },
                    "functions": [ response_schema ],
                    "messages": [
                        { "role": "system","content":request_prompt},
                        { "role" : "user","content":json.dumps(payload, ensure_ascii=False)}
                    ],
                    "function_call": { "name": "selectRegion" }
                }
            }, ensure_ascii=False) + "\n")
    print("✔ built batch input: ", batch_path)
    return batch_path

Load the region standardisation prompt and function-calling schema using the same helpers as Stage 3. The prompt instructs the LLM to map each raw Ukrainian region string to a standardised English oblast name from a fixed allowed list.

In [ ]:
prompt = st3.read_classification_prompt(g.Config.STAGE4_5_SELECT_REGION_PROMPT)
scheme = st3.read_classification_schema(g.Config.STAGE4_5_SELECT_REGION_SCHEME)

> Batch files created for OpenAPI are separated custom by analytics for gpt-4o model batch size limitations.

Check total number of new region strings to decide how to split them into batch chunks. Set `fr` (from index), `to` (to index), and `today` (date tag for the filename) before writing the batch file.

In [ ]:
new_regions_db.shape[0]

In [ ]:
fr = 0
to = 20
today = "010124"

Write the JSONL batch input file for the current slice `new_regions_db[fr:to]`. The output file is saved to `data/data-pipeline/stage_04_5/input/` and its path is stored in `input_batch_path` for the next step. Repeat for each slice until all new regions are covered.

In [ ]:
input_batch_path = write_region_batch_file(g.Config.STAGE4_5_INPUT_BATCH_PATH, f"{fr}_{to}_{today}", new_regions_db[fr:to], scheme, prompt, "gpt-4.1")

## 4.5.5. Create Job

Upload the JSONL file to the OpenAI Files API and submit it as a Batch job with a 24-hour completion window. The returned `job` object contains the `job.id` needed to poll for status. This cell submits **one** batch — repeat for each input slice.

In [ ]:
batch_file = client.files.create(
     file=open(input_batch_path, "rb"),
     purpose="batch")

job = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h")

Display the job ID. Copy this value and add it to the `job_ids` list in the next section to track it for download.

In [ ]:
job.id

## 4.5.6. Creating list of batch job_ids

Accumulate all submitted batch job IDs in the `job_ids` list. After each job completes (typically within minutes to a few hours), uncomment its ID line to include it in the download loop. The `job_id = job.id` cell below captures the most recently created job for convenience.

In [ ]:
job_ids = [
    #'batch_6851621deccc8190a9800829199532ed',  # 1-100 done
    #'batch_68516457699081909d3d32639cae945a',  # 100-120 done
    #'batch_68516502164c8190a1a73932fc154ea9',  # 120-150 done
    #'batch_6851660b2dac8190b51bfd4601850946',  # 150-160 done
    #'batch_68516680d13c8190a174fc7b99274313',  # 160-175 done
    #'batch_685167ad52d881908c180c3c7eb569a2',  # 175-275 done
    #'batch_68516cb4c12c81908aa879839062555e',  # 275-325 done
    #'batch_68517700d4e481909bd58ce082681c16',  # 325-425 done
    #'batch_6851790c5d508190baae21fb86888cdd',  # 425-450 done
    #'batch_68517d7a5c1c819098b12ced6d1b8a51',  # 450-475 done
    #'batch_68517de86f04819087183be1ffe5f44b',  # 475-525 done
    #'batch_68517e7460008190af55eff4333310ab',  # 525-575 done
    #'batch_68517f84b59081909f5bd1267dc3c88b',  # 575-590 done
    #'batch_6851808b0e6c8190b2616836432c7d68',  # 590-595 done
   # 'batch_6852548f703c8190806698b2d8f159e0',  # 595-600 done
   # 'batch_685255435f388190ab1f1d2c52e3dafe',  # 600-610 done
    #'batch_6852603984fc8190967f98aaf859eef5',  # 610-620 done
  #  'batch_685260bf96508190b2e635d58c428a8a',  # 620-700 done
    #'batch_6852615f556c81909744982a9754e120',  # 700-800 done
   # 'batch_685261dec1a48190b1df3dc60f08d9be',   # final batch
    #'batch_6885ca33043c8190aa58dd6c3206141b',   # 0-100 270726
    #'batch_6885cd71769081908ef180cf9d088b75',   #100-200 270726
  #  'batch_6885ce5bd4308190936930c5e0e8012c',    #200-300 270726
    #'batch_6885ce9b8e0481908163d3cd4f898c3e',     #300-400 270726
  #  'batch_6885ced61e64819096b9e8d27b5906d9',    #400-500 270726
  #  'batch_6885cfffa0e88190bc9f5b09a3bced31',    #500-700 270726
  #  'batch_6885d0308ac8819089ebe8ea73b03a23',    #700-1000 270726
  #  'batch_6885d0d167c8819089596d2eb551ced2',   #1000-1300 270726
  #  'batch_6885d1083b848190a0747bec538b75ea',    #1300-1500 270726
   # 'batch_6885d1cec5b88190b9f9632ddd4af1b5',   #1500-1800 270726
   # 'batch_6885d225b8508190bedc3e5fc8540235',   #1800-1900 270726
  #  'batch_6885d29df0bc8190b0e5d59d5daa273e',   #1900-2000 270726
  #  'batch_6885d331e13481909149b98ea1d0d215',   #2000-2100 270726
  #  'batch_6885d365a7fc8190abd94330ce6406f9',   #2100-2200 270726
  #  'batch_6885e2049a408190b34aad75d0bf7b14',    #2200-2250 270726
  #  'batch_6885e356fbb88190a2d8eab879dec4c0',   #2250-2350 270726
  #  'batch_6885e3bec88881909a8e2c769825e621',    #2350-2400 270726
  #  'batch_6885e3f639a081909523464b92c58cb4',    #2400-2700 270726
   # 'batch_6885e47e139c8190a7fdc1a05079b631',    #2700-2750 270726
    #'batch_6885e4a976a08190914695275b5224a0',   #2750-2850 270726
    #'batch_6885e51015d481909c049b1957e1d91b',    #2850-2900 270726
   # 'batch_6885e5d7a0b48190a72d6797665b78c7',   #2900-3000 270726
   # 'batch_6885e5f835b4819099054f4762f1da2b',    #3000-3100 270726
   # 'batch_6885e727677c8190bc5bc5428557e4c2',    #3100-3200 270726
   # 'batch_6885e752765c8190a6820f8628cd5c0d',    #3100-3200 270726
   # 'batch_6885e7b52b8081909327d35d9d73fb4e', #3200-3400 270726
   # 'batch_6885e7dca6c48190a4c3561e7b7f48c2', #3400-3500 270726
    #'batch_6885e87256a0819089728b65cddc4b22', #3500-3600 270726
    #'batch_6885e8acf8448190baa0d661d685ae75', #3600-3700 270726
    #'batch_6885e8ead6f88190ad1456e621389a72', #3700-3800 270726
    #'batch_6885e919de4081908bff705f75ad1f99', #3800-3900 270726
    #'batch_6885ea2d6598819080feebb5d6ecd161', #final
   # 'batch_68e1951711488190bb6bf5432deffc68', #2021-2022 classify start 0-1000
   # 'batch_68e19897729c819084e2657f80ae2ea7',
    #'batch_68e19929a7888190a3eb017ab8c36c8f',#'batch_68e1998d35e08190aa6fa1119ffea708',
    #'batch_68e199a1071481909fb2742b4e4525d2',
    #'batch_68e19bad1ba081909fdbcf200b066a4d', #5-7K
    #'batch_68e20ac0aec08190b3da8aae46a1f6d1', #7-9K
    #'batch_68e20b1561d881908b2587c69f65717b', #9-10K
    #'batch_68e2129722508190a0d8d2023219c9de', #10-12
    #'batch_68e212af3c9c8190940bf4a4455010a3',  #12-14
    #'batch_68e21447559c81909a51bfadc5dd8d93',
    #'batch_68e2146029b48190bb5160fcf02a604a', #16-18
    #'batch_68e216811f2c81908f17bfac75322e79', #18-20
    #'batch_68e21a20f8bc8190b0453f780124715c' #20-22 - final
    'batch_69c082a1f0bc819090b98c699c5bc1f2'
]


In [ ]:
job_id = job.id

## 4.5.6. Download output batches

Poll each job in `job_ids`. When a job's status is `completed`, download the output JSONL from the OpenAI Files API and save it to `data/data-pipeline/stage_04_5/output/` using the batch job ID as the filename. Rerun this cell until all jobs are downloaded.

In [ ]:
for job_id in job_ids:
    batch = client.batches.retrieve(job_id)
    if batch.status == "completed":
        result_file_id = batch.output_file_id
        result = client.files.content(result_file_id).content
        with open(os.path.join(g.Config.STAGE4_5_OUTPUT_BATCH_PATH, f"{job_id}.jsonl"), 'wb') as file:
            file.write(result)
        print("Done: comment this :)")
    else:
        print("Status: ", batch.status)

## 4.5.7. Combining all output files

Combine all downloaded output JSONL files from `data/data-pipeline/stage_04_5/output/` into a single `combined_output.jsonl`. This merged file is used in the next step to parse all results at once, regardless of how many batch jobs were submitted.

In [ ]:
# Create a list of files in a predefined folder (using Config.STAGE4_5_OUTPUT_BATCH_PATH)
folder_path = g.Config.STAGE4_5_OUTPUT_BATCH_PATH
output_file = g.Config.STAGE4_5_COMBINED_OUTPUT_PATH
_files = 0
_lines = 0

with open(output_file, 'w', encoding='utf-8') as outfile:
    for filename in os.listdir(folder_path):
        fpath = os.path.join(folder_path, filename)
        _files += 1
        file_path = os.path.join(folder_path, filename)
        print(f"Processing file: {file_path}")
        with open(file_path, 'r', encoding='utf-8') as infile:
            for line in infile:
                _lines += 1
                outfile.write(line)
                print(f"File #{_files}, line #{_lines}")

print(f"All .jsonl files combined into: {output_file}")

## 4.5.8. Extract results and save

Parse `combined_output.jsonl` — navigate the nested Batch API response structure to extract `original` (raw region string) and `region` (standardised English oblast name) from each function-call result. Deduplicate on `original`, inspect the region distribution, then **save** the final merged DB back to `region_db.pkl`.

> The last cell (`final_df.to_pickle(...)`) is commented out as a safety guard — uncomment it only when you are satisfied with the results.

**Output columns of `region_db.pkl`:**

| Column | Description |
|--------|-------------|
| `original` | Raw region string from the job board (Cyrillic / transliterated) |
| `region` | Standardised English oblast name (e.g. `"Kyiv Oblast"`) |

In [ ]:
with open(output_file, 'r') as file:
    data = [json.loads(line) for line in file]

    records = []
    for entry in data:
        choices = entry.get('response', {}).get('body', {}).get('choices', [])
        if choices:
            message = choices[0].get('message', {})
            function_call = message.get('function_call', {})
            arguments_json = function_call.get('arguments')

            if arguments_json:
                arguments = json.loads(arguments_json)
                records.append({
                    'original': arguments.get('original'),
                    'city': arguments.get('city'),
                    'district': arguments.get('district'),
                    'region': arguments.get('region'),
                    'country': arguments.get('country'),
                    "latitude": arguments.get("latitude"),
                    "longitude": arguments.get("longitude",),
                    'is_remote': arguments.get('is_remote')})

    final_df = pd.DataFrame(records)
    final_df = final_df.astype({
        "original" : str,
        "city" : str,
        "region" : str,
        "district" : str,
        "country" : str,
        "latitude" : str,
        "longitude" : str,
        "is_remote" : str
    })

final_df.tail()

In [ ]:
final_df.drop_duplicates(subset=['original'], keep="first", inplace=True)
final_df

In [ ]:
region_counts = final_df['region'].value_counts()
region_counts_df = region_counts.reset_index()
region_counts_df.columns = ['Region', 'Count']
region_counts_df = region_counts_df.sort_values('Region')
region_counts_df

In [ ]:
final_df.to_pickle(g.Config.STAGE4_5_REGION_DB_PATH)

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.